# Setting the Environment

Install the Ollama in the terminal using Linux command.

curl -fsSL https://ollama.com/install.sh | sh

Pull Llama3 model

ollama serve & ollama pull llama3.2:1b

In [2]:
!pip install pymupdf langchain-text-splitters langchain-huggingface langchain-community langchain-chroma langchain-ollama ragas

### Extract Document

In [3]:

file_paths = [
  "./dataset/PERATURAN REKTOR NO 11 TAHUN 2020 TENTANG PANDUAN MAGANG ITK.pdf",
  "./dataset/PERATURAN REKTOR NO 12 TAHUN 2020 TENTANG PANDUAN TUGAS AKHIR.pdf",
  "./dataset/PERATURAN REKTOR NO 13 TAHUN 2020 TENTANG PANDUAN KERJA PRAKTIK.pdf",
  "./dataset/Peraturan Rektor Nomor 10 Tahun 2021 Tentang PENYELENGGARAAN KEGIATAN MERDEKA BELAJAR - KAMPUS MERDEKA.pdf"]



In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.schema import Document
def extract_text_from_pdf(file_path) -> list:
    loader = PyMuPDFLoader(file_path)
    text_pages = []
    for page in loader.lazy_load():
        text = page.page_content
        if text.strip():
            text_pages.append(Document(page_content=text, metadata={
                "page": page.metadata["page"],
                "source": file_path
            }))

    return text_pages

In [5]:
import pymupdf4llm
from langchain.schema import Document
import re
def extract_markdown_from_pdf(file_path)-> list[Document]:
  md_chunks = pymupdf4llm.to_markdown(file_path,
                                  # image_path="gambar",
                                  # image_format="png",
                                  # write_images=True,
                                  page_chunks=True,
                                  # dpi=300,
                                  )
  docs = []
  for i, chunk in enumerate(md_chunks, start=1):
    page = i
    text = chunk["text"]

    image_paths = re.findall(r'\[!\[.*?\]\((.*?)\)\]', text)
    docs.append(Document(page_content=text, metadata={"page": page, "source": file_path, "images":image_paths}))
  return docs

ModuleNotFoundError: No module named 'pymupdf4llm'

In [ ]:
import pathlib
for i, file_path in enumerate(file_paths):
  md_text = extract_markdown_from_pdf(file_path=file_path)
  md_text = "\n\n".join([doc.page_content for doc in md_text])
  pathlib.Path(f"output_{i+1}.md").write_bytes(md_text.encode())



### Chunking Documents

Metode pembuatan chunk dokumen berdasarkan kalimat

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain.schema import Document
import re

def clean_text(text: str) -> str:
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunking_document(documents, chunk_size=128, chunk_overlap=0):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "]
    )
    chunked_docs = []

    for doc in documents:
        cleaned_text = clean_text(doc.page_content)
        splits = text_splitter.split_text(cleaned_text)

        for i, chunk in enumerate(splits):
            chunked_docs.append(Document(
                page_content=chunk,
                metadata={
                    **doc.metadata,
                    "chunk_id": i
                }
            ))

    return filter_complex_metadata(chunked_docs)


### Storing & Indexing Vector Databases

In [7]:
import torch
print( torch.cuda.is_available() )

True


In [8]:
import os
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_chroma import Chroma
import logging

# Konfigurasi logging
logging.basicConfig(level=logging.INFO)

def init_embedding(model_name="firqaaa/indo-sentence-bert-base", device: str | None = None):
    """
    Initialize embedding model, with optional device specification.

    Args:
        model_name (str): name of the embedding model.
        device (str | None): e.g. "cuda", "cpu", or None to auto-detect.

    Returns:
        An embeddings instance (OllamaEmbeddings or HuggingFaceEmbeddings).
    """
    try:
        # Auto-detect device if tidak diset
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        logging.info(f"Initializing embeddings for '{model_name}' on device: {device}")

        if "llama" in model_name:
            # OllamaEmbeddings tidak punya opsi device (ditangani oleh server Ollama)
            return OllamaEmbeddings(model=model_name, base_url="http://localhost:11434")
        else:
            # Untuk HF, pass device via model_kwargs
            return HuggingFaceEmbeddings(
                model_name=model_name,
                model_kwargs={"device": device},
            )
    except Exception as e:
        logging.error(f"Error occurred while initializing embedding: {e}", exc_info=True)
        raise

def init_llm(model="llama3.2:1b"):
  return Ollama(model=model, base_url="http://localhost:11434")

def init_vector_db(embedding_func,name):
  if os.path.exists("./vector_db"+ name):
    shutil.rmtree("./vector_db"+ name)
  return Chroma(
    embedding_function=embedding_func,
    collection_name=name

  )

In [9]:

from uuid import uuid4
from langchain.schema import Document
from langchain_chroma import Chroma

def storing_to_vector_db(vector_db:Chroma,documents:list[Document]):
  uuids = [str(uuid4()) for _ in range(len(documents))]
  print(f"Adding {len(documents)} documents to the vector database.")
  vector_db.add_documents(documents= documents, ids=uuids)
  print(f"Successfully added {len(documents)} documents to the vector database with {len(uuids)} unique IDs.")


In [10]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

def prompt_template(question, chunks):
    context = "\n\n".join([
        f"[HALAMAN {doc.metadata.get('page', '?')} - CHUNK {doc.metadata.get('chunk_id', '?')}]:\n{doc.page_content}"
        for doc in chunks
  ])
    template = """
Kamu adalah asisten akademik cerdas dari Institut Teknologi Kalimantan (ITK). Jawablah pertanyaan pengguna berdasarkan dokumen yang tersedia. Jika tidak ditemukan jawabannya dalam dokumen, katakan bahwa informasi tersebut tidak tersedia.

Konteks:
{context}

Pertanyaan pengguna:
{question}

Instruksi:
- Gunakan konteks untuk menjawab pertanyaan dengan jelas dan ringkas.
- Jika informasi tidak ditemukan dalam konteks, balas dengan: "Maaf, informasi tersebut tidak tersedia dalam data akademik ITK yang saya miliki."
- Jangan membuat jawaban dari asumsi atau tebakan.
    """
    prompt =  PromptTemplate(
        input_variables=["question", "context"],
        template=template
    )
    return prompt.format(question=question, context=context)
def search_with_neighbors(vector_db:Chroma,query, k=1, neighbor_window=1):
  retrieve_docs =  vector_db.similarity_search(query,k=k)
  temp = retrieve_docs
  # return retrieve_docs
  all_docs = vector_db._collection.get(include=["documents", "metadatas"])

  docs = [Document(page_content=d, metadata=m) for d, m in zip(all_docs["documents"], all_docs["metadatas"])]

  neighbors = []
  seen_indexes = set()

  for doc in retrieve_docs:  # ✅ tidak unpack
      idx = next((i for i, d in enumerate(docs) if d.page_content == doc.page_content), None)
      if idx is None:
          continue

      start = max(0, idx - neighbor_window)
      end = min(len(docs), idx + neighbor_window + 1)

      for i in range(start, end):
          if i not in seen_indexes:
              neighbors.append(docs[i])
              seen_indexes.add(i)

  return {
      "context_with_neighbors": neighbors,
      "context": temp
  }
     


def generate_response(llm: OllamaLLM, vector_db:Chroma,query):

  chunks = search_with_neighbors(vector_db,query, k=5, neighbor_window=5)
  messages = prompt_template(query,chunks)
  response = llm.invoke(messages)
  return {"question": query,"answer": response, "context": chunks}







In [11]:
embedding_func = init_embedding()
vector_db = init_vector_db(embedding_func,"indo_sentence_bert_1")
llm = init_llm(model="llama3.2:3b")


INFO:root:Initializing embeddings for 'firqaaa/indo-sentence-bert-base' on device: cuda
INFO:datasets:PyTorch version 2.7.0+cu128 available.
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: firqaaa/indo-sentence-bert-base


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/709k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
/tmp/ipykernel_149280/3275514540.py:43: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  return Ollama(model=model, base_url="http://localhost:11434")


In [ ]:
md_kp = extract_text_from_pdf(file_path=file_paths[2])
chunks_md_kp = chunking_document(md_kp,chunk_size=250,chunk_overlap=20)
storing_to_vector_db(vector_db,chunks_md_kp)

In [ ]:
test_set[17]["question"]

In [ ]:
retrieve_docs = search_with_neighbors(vector_db,test_set[18]["question"],k=5)
retrieve_docs

In [ ]:
len(retrieve_docs)

In [ ]:
generate_response(llm,vector_db,"Berapa waktu pelaksanaan KP di ITK?")

In [12]:
from langchain_core.prompts import PromptTemplate
def prompt_template_test():
  template = """
Kamu adalah asisten akademik cerdas dari Institut Teknologi Kalimantan (ITK). Jawablah pertanyaan pengguna berdasarkan dokumen yang tersedia. Jika tidak ditemukan jawabannya dalam dokumen, katakan bahwa informasi tersebut tidak tersedia.

Konteks:
{context}

Pertanyaan pengguna:
{question}

Instruksi:
- Gunakan konteks untuk menjawab pertanyaan dengan jelas dan ringkas.
- Konteks-konteks tersebut merupakan konteks dokumen  dilingkungan Institut Teknologi Kalimantan atau ITK yang hanya relevan untuk civitas akademika ITK
- Jika informasi tidak ditemukan dalam konteks, balas dengan: "Maaf, informasi tersebut tidak tersedia dalam data akademik ITK yang saya miliki."
- Jangan membuat jawaban dari asumsi atau tebakan.
    """
  return PromptTemplate(
        input_variables=["question", "context"],
        template=template
    )


In [13]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from typing import List,Optional

class NeighborRetriever(BaseRetriever):
    vector_db: Chroma
    k: int = 1
    neighbor_window: int = 1
    real_context: Optional[List[Document]] = None  # untuk menyimpan hasil asli

    def _get_relevant_documents(self, query: str) -> List[Document]:
        result = search_with_neighbors(
            vector_db=self.vector_db,
            query=query,
            k=self.k,
            neighbor_window=self.neighbor_window
        )
        self.real_context = result["context"] 
        return result["context_with_neighbors"] 

    def get_real_context(self) -> Optional[List[Document]]:
        return self.real_context


In [14]:
import os
from langchain.chains import RetrievalQA
import pandas as pd


def create_rag_chain(docs : list[Document], llm: OllamaLLM, embedding_func, collection_name,k=1, neighbor_window=1):
  vector_db = Chroma.from_documents(docs, embedding_func, collection_name=collection_name)
  retriever = NeighborRetriever(vector_db=vector_db, k=k, neighbor_window=neighbor_window)
  return RetrievalQA.from_chain_type( llm=llm, retriever=retriever,chain_type="stuff",chain_type_kwargs={"prompt":prompt_template_test(),},return_source_documents=True)


In [15]:
from ragas.prompt import PydanticPrompt
from ragas.metrics._faithfulness import NLIStatementInput, NLIStatementOutput, StatementFaithfulnessAnswer, StatementGeneratorInput, StatementGeneratorOutput

class NLIStatementPrompt(PydanticPrompt[NLIStatementInput, NLIStatementOutput]):
    instruction = (
        "Tugasmu adalah menilai kesetiaan sebuah pernyataan terhadap konteks yang diberikan. "
        "Untuk setiap pernyataan, berikan verdict 1 jika pernyataan tersebut bisa langsung disimpulkan "
        "berdasarkan konteks, atau 0 jika tidak."
    )
    input_model = NLIStatementInput
    output_model = NLIStatementOutput
    examples = [
        (
            NLIStatementInput(
                context=(
                    "John adalah mahasiswa di Universitas XYZ. Dia mengambil jurusan Ilmu Komputer. "
                    "Dia sedang mengikuti beberapa mata kuliah semester ini, termasuk Struktur Data, Algoritma, dan Manajemen Database. "
                    "John adalah mahasiswa yang rajin dan banyak menghabiskan waktu belajar dan mengerjakan tugas. "
                    "Dia sering begadang di perpustakaan untuk mengerjakan proyeknya."
                ),
                statements=[
                    "John mengambil jurusan Biologi.",
                    "John sedang mengikuti mata kuliah Kecerdasan Buatan.",
                    "John adalah mahasiswa yang berdedikasi.",
                    "John memiliki pekerjaan paruh waktu.",
                ],
            ),
            NLIStatementOutput(
                statements=[
                    StatementFaithfulnessAnswer(
                        statement="John mengambil jurusan Biologi.",
                        reason="Jurusan John secara eksplisit disebutkan Ilmu Komputer. Tidak ada informasi yang menyebutkan Biologi.",
                        verdict=0,
                    ),
                    StatementFaithfulnessAnswer(
                        statement="John sedang mengikuti mata kuliah Kecerdasan Buatan.",
                        reason=(
                            "Konteks menyebutkan mata kuliah yang diambil John, dan Kecerdasan Buatan tidak ada di dalamnya. "
                            "Jadi tidak bisa disimpulkan dia mengambil mata kuliah itu."
                        ),
                        verdict=0,
                    ),
                    StatementFaithfulnessAnswer(
                        statement="John adalah mahasiswa yang berdedikasi.",
                        reason=(
                            "Konteks menyatakan John banyak menghabiskan waktu belajar dan mengerjakan tugas, "
                            "serta sering begadang di perpustakaan, yang mengindikasikan dedikasi."
                        ),
                        verdict=1,
                    ),
                    StatementFaithfulnessAnswer(
                        statement="John memiliki pekerjaan paruh waktu.",
                        reason="Tidak ada informasi dalam konteks mengenai pekerjaan paruh waktu John.",
                        verdict=0,
                    ),
                ]
            ),
        ),
        (
            NLIStatementInput(
                context="Fotosintesis adalah proses yang digunakan oleh tumbuhan, alga, dan beberapa bakteri untuk mengubah energi cahaya menjadi energi kimia.",
                statements=[
                    "Albert Einstein adalah seorang jenius.",
                ],
            ),
            NLIStatementOutput(
                statements=[
                    StatementFaithfulnessAnswer(
                        statement="Albert Einstein adalah seorang jenius.",
                        reason="Konteks dan pernyataan tidak terkait.",
                        verdict=0,
                    )
                ]
            ),
        ),
    ]
class StatementGeneratorPrompt(
    PydanticPrompt[StatementGeneratorInput, StatementGeneratorOutput]
):
    instruction = (
    "Diberikan sebuah pertanyaan dan jawaban, analisis kompleksitas setiap kalimat dalam jawaban tersebut. "
    "Pecah setiap kalimat menjadi satu atau lebih pernyataan yang mudah dipahami. "
    "Pastikan tidak ada kata ganti (pronoun) yang digunakan dalam pernyataan manapun. "
    "Jawaban harus berupa **JSON yang valid saja**, tanpa penjelasan tambahan, "
    "dengan format sesuai schema berikut:\n"
    '{ "statements": ["...","..."] }'
)

    input_model = StatementGeneratorInput
    output_model = StatementGeneratorOutput
    examples = [
        (
            StatementGeneratorInput(
                question="Siapa Albert Einstein dan apa yang paling dikenal darinya?",
                answer=(
                    "Dia adalah seorang fisikawan teoretis kelahiran Jerman yang secara luas diakui sebagai salah satu fisikawan terbesar dan paling berpengaruh sepanjang masa. "
                    "Dia paling dikenal karena mengembangkan teori relativitas, dan juga memberikan kontribusi penting pada pengembangan teori mekanika kuantum."
                ),
            ),
            StatementGeneratorOutput(
                statements=[
                    "Albert Einstein adalah seorang fisikawan teoretis kelahiran Jerman.",
                    "Albert Einstein secara luas diakui sebagai salah satu fisikawan terbesar dan paling berpengaruh sepanjang masa.",
                    "Albert Einstein paling dikenal karena mengembangkan teori relativitas.",
                    "Albert Einstein juga memberikan kontribusi penting pada pengembangan teori mekanika kuantum.",
                ]
            ),
        )
    ]


In [16]:
from ragas.metrics import Faithfulness
from langchain.prompts import PromptTemplate

faithfulness_indonesia = Faithfulness(
    name="faithfulness_indonesia",
    nli_statements_prompt= NLIStatementPrompt(language="indonesian"),
    statement_generator_prompt= StatementGeneratorPrompt(language="indonesian")
)
faithfulness_indonesia.get_prompts()



{'n_l_i_statement_prompt': NLIStatementPrompt(instruction=Tugasmu adalah menilai kesetiaan sebuah pernyataan terhadap konteks yang diberikan. Untuk setiap pernyataan, berikan verdict 1 jika pernyataan tersebut bisa langsung disimpulkan berdasarkan konteks, atau 0 jika tidak., examples=[(NLIStatementInput(context='John adalah mahasiswa di Universitas XYZ. Dia mengambil jurusan Ilmu Komputer. Dia sedang mengikuti beberapa mata kuliah semester ini, termasuk Struktur Data, Algoritma, dan Manajemen Database. John adalah mahasiswa yang rajin dan banyak menghabiskan waktu belajar dan mengerjakan tugas. Dia sering begadang di perpustakaan untuk mengerjakan proyeknya.', statements=['John mengambil jurusan Biologi.', 'John sedang mengikuti mata kuliah Kecerdasan Buatan.', 'John adalah mahasiswa yang berdedikasi.', 'John memiliki pekerjaan paruh waktu.']), NLIStatementOutput(statements=[StatementFaithfulnessAnswer(statement='John mengambil jurusan Biologi.', reason='Jurusan John secara eksplisit 

In [17]:
from ragas.metrics._answer_relevance import  ResponseRelevanceOutput, ResponseRelevanceInput

class ResponseRelevancePrompt(
    PydanticPrompt[ResponseRelevanceInput, ResponseRelevanceOutput]
):
    instruction = """
Buatlah sebuah pertanyaan yang relevan untuk jawaban yang diberikan dan identifikasi apakah jawaban tersebut bersifat tidak berkomitmen.  
Berikan nilai `noncommittal` sebagai 1 jika jawaban bersifat tidak berkomitmen, dan 0 jika jawaban bersifat berkomitmen.  
Jawaban yang tidak berkomitmen adalah jawaban yang menghindar, samar, atau ambigu, seperti "Saya tidak tahu" atau "Saya tidak yakin".

**Instruksi Penting:**
- Output harus berupa JSON yang valid.
- Jangan sertakan penjelasan, narasi, atau teks selain JSON.
- Format JSON yang diharapkan:  
```json
{
  "question": "Pertanyaan terkait jawaban",
  "noncommittal": 0
}"""
 
    input_model = ResponseRelevanceInput
    output_model = ResponseRelevanceOutput

    examples = [
        (
            ResponseRelevanceInput(
                response="""Albert Einstein lahir di Jerman.""",
            ),
            ResponseRelevanceOutput(
                question="Di mana Albert Einstein lahir?",
                noncommittal=0,
            ),
        ),
        (
            ResponseRelevanceInput(
                response="""Saya tidak tahu tentang fitur revolusioner dari smartphone yang ditemukan pada tahun 2023 karena saya tidak memiliki informasi setelah tahun 2022.""",
            ),
            ResponseRelevanceOutput(
                question="Apa fitur revolusioner dari smartphone yang ditemukan pada tahun 2023?",
                noncommittal=1,
            ),
        ),
    ]



In [18]:
from ragas.metrics import AnswerRelevancy
from ragas.llms import llm_factory
from ragas.llms import LangchainLLMWrapper
answer_relevancy_indonesia = AnswerRelevancy(
    name="answer_relevancy_indonesia",question_generation= ResponseRelevancePrompt(
        language="indonesian"
    )
)
answer_relevancy_indonesia.get_prompts()

{'response_relevance_prompt': ResponseRelevancePrompt(instruction=
 Buatlah sebuah pertanyaan yang relevan untuk jawaban yang diberikan dan identifikasi apakah jawaban tersebut bersifat tidak berkomitmen.  
 Berikan nilai `noncommittal` sebagai 1 jika jawaban bersifat tidak berkomitmen, dan 0 jika jawaban bersifat berkomitmen.  
 Jawaban yang tidak berkomitmen adalah jawaban yang menghindar, samar, atau ambigu, seperti "Saya tidak tahu" atau "Saya tidak yakin".
 
 **Instruksi Penting:**
 - Output harus berupa JSON yang valid.
 - Jangan sertakan penjelasan, narasi, atau teks selain JSON.
 - Format JSON yang diharapkan:  
 ```json
 {
   "question": "Pertanyaan terkait jawaban",
   "noncommittal": 0
 }, examples=[(ResponseRelevanceInput(response='Albert Einstein lahir di Jerman.'), ResponseRelevanceOutput(question='Di mana Albert Einstein lahir?', noncommittal=0)), (ResponseRelevanceInput(response='Saya tidak tahu tentang fitur revolusioner dari smartphone yang ditemukan pada tahun 2023 k

In [19]:
from dataclasses import dataclass, field
import typing as t
import numpy as np
from ragas.metrics.base import MetricType, MetricWithLLM, SingleTurnMetric
from ragas.dataset_schema import SingleTurnSample
from langchain_core.callbacks import Callbacks
from langchain_core.prompt_values import StringPromptValue
import logging

logger = logging.getLogger(__name__)

@dataclass
class ContextRelevanceIndonesia(MetricWithLLM, SingleTurnMetric):
    """
    Parameter:
    Mengukur relevansi konteks yang diambil (retrieved contexts) berdasarkan pertanyaan pengguna.

    Input:
        data: daftar Dict dengan kunci: user_input, retrieved_contexts
    Output:
        0.0: konteks tidak relevan sama sekali dengan pertanyaan
        0.5: konteks relevan sebagian
        1.0: konteks sepenuhnya relevan
    """

    name: str = field(default="context_relevance_indonesia", repr=True)  # type: ignore
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {
            MetricType.SINGLE_TURN: {
                "user_input",
                "retrieved_contexts",
            },
        }
    )

    # Template instruksi dalam Bahasa Indonesia
    template_relevance1 = (
        "### Instruksi\n\n"
        "Anda adalah seorang ahli kelas dunia yang ditugaskan untuk menilai skor relevansi dari sebuah konteks "
        "untuk menjawab sebuah pertanyaan.\n"
        "Tugas Anda adalah menentukan apakah konteks tersebut berisi informasi yang sesuai untuk menjawab pertanyaan.\n"
        "Jangan gunakan pengetahuan Anda sebelumnya tentang pertanyaan tersebut.\n"
        "Gunakan hanya informasi yang tertulis di dalam konteks dan pertanyaan.\n"
        "Ikuti petunjuk berikut:\n"
        "0. Jika konteks tidak berisi informasi yang relevan untuk menjawab pertanyaan, beri skor 0.\n"
        "1. Jika konteks berisi sebagian informasi yang relevan, beri skor 1.\n"
        "2. Jika konteks berisi informasi yang sepenuhnya relevan, beri skor 2.\n"
        "Anda harus memberikan skor relevansi berupa 0, 1, atau 2. Jangan berikan penjelasan apapun.\n\n"
        "### Pertanyaan: {query}\n\n"
        "### Konteks: {context}\n\n"
        "Jangan berikan penjelasan.\n"
        "Setelah menganalisis konteks dan pertanyaan, skor relevansi adalah "
    )

    template_relevance2 = (
        "Sebagai seorang ahli yang dirancang khusus untuk menilai tingkat relevansi sebuah konteks terhadap pertanyaan, "
        "tugas saya adalah menentukan sejauh mana konteks tersebut menyediakan informasi yang dibutuhkan untuk menjawab pertanyaan.\n"
        "Saya hanya akan menggunakan informasi yang tersedia di dalam konteks dan pertanyaan, tanpa menggunakan pengetahuan sebelumnya.\n\n"
        "Berikut petunjuk yang akan saya ikuti:\n"
        "* Jika konteks tidak berisi informasi relevan, saya akan menjawab dengan skor relevansi 0.\n"
        "* Jika konteks hanya sebagian relevan, saya akan memberikan skor 1.\n"
        "* Jika konteks sepenuhnya relevan, saya akan memberikan skor 2.\n\n"
        "### Pertanyaan: {query}\n\n"
        "### Konteks: {context}\n\n"
        "Jangan mencoba menjelaskan.\n"
        "Berdasarkan Pertanyaan dan Konteks yang diberikan, skor relevansinya adalah ["
    )

    retry = 5  # Jumlah percobaan ulang jika skor tidak ditemukan dalam 8 token pertama.

    def process_score(self, response):
        for i in [2, 1, 0]:
            if str(i) in response:
                return i / 2
        return np.nan

    def average_scores(self, score0, score1):
        if score0 >= 0 and score1 >= 0:
            return (score0 + score1) / 2
        return max(score0, score1)

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks: Callbacks
    ) -> float:
        assert self.llm is not None, "Model LLM belum diatur"
        assert sample.user_input is not None, "Input pengguna belum diatur"
        assert sample.retrieved_contexts is not None, "Konteks belum diatur"

        if (sample.user_input.strip() == "") or (
            "\n".join(sample.retrieved_contexts).strip() == ""
        ):
            return 0.0
        if sample.user_input.strip() == "\n".join(sample.retrieved_contexts).strip():
            return 0.0
        if "\n".join(sample.retrieved_contexts).strip() in sample.user_input.strip():
            return 0.0

        try:
            score0 = score1 = np.nan
            for retry in range(self.retry):
                formatted_prompt = StringPromptValue(
                    text=self.template_relevance1.format(
                        query=sample.user_input,
                        context="\n".join(sample.retrieved_contexts)[:7000],
                    )
                )
                resp = await self.llm.agenerate_text(
                    formatted_prompt,
                    n=1,
                    temperature=0.1,
                )
                score0 = self.process_score(resp.generations[0][0].text)
                if score0 == score0:
                    break
                logger.warning(f"Percobaan ulang: {retry}")

            for retry in range(self.retry):
                formatted_prompt = StringPromptValue(
                    text=self.template_relevance1.format(
                        query=sample.user_input,
                        context="\n".join(sample.retrieved_contexts)[:7000],
                    )
                )
                resp = await self.llm.agenerate_text(
                    formatted_prompt,
                    n=1,
                    temperature=0.1,
                )
                score1 = self.process_score(resp.generations[0][0].text)
                if score1 == score1:
                    break
                logger.warning(f"Percobaan ulang: {retry}")

            return self.average_scores(score0, score1)

        except Exception as e:
            print(f"Terjadi kesalahan: {e}. Skor akan diberikan NaN.")
            return np.nan

context_relevance_indonesia = ContextRelevanceIndonesia()
context_relevance_indonesia



ContextRelevanceIndonesia(_required_columns={<MetricType.SINGLE_TURN: 'single_turn'>: {'user_input', 'retrieved_contexts'}}, name='context_relevance_indonesia', llm=None, output_type=None)

In [ ]:
import time
from datasets import Dataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import ContextRelevance
from langchain_community.vectorstores.utils import filter_complex_metadata
import pandas as pd
from ragas.run_config import RunConfig

def run_ragas_full_eval(filepaths, questions, ground_truths, llm_types: list, embedding_types: list):
    for llm_type in llm_types:
        for embedding_type in embedding_types:
            results = []
            chunks = []

            # Ekstraksi dan chunking semua dokumen PDF
            for file_path in filepaths:
                text_pdf = extract_text_from_pdf(file_path=file_path)
                new_chunks = chunking_document(text_pdf, chunk_size=250, chunk_overlap=50)
                chunks.extend(new_chunks)

            # Filter chunk dari metadata kompleks
            filtered_chunks = filter_complex_metadata(chunks)

            print(f"Load LLM {llm_type['type']}")
            llm = init_llm(model=llm_type['type'])

            print(f"Load Embedding {embedding_type['type']}")
            embedder = init_embedding(model_name=embedding_type['type'])\

            # Buat RAG chain
            qa = create_rag_chain(filtered_chunks, llm, embedder, f"{llm_type['name']}_{embedding_type['name']}_k5_nw_3_5", k=5, neighbor_window=3)

            # Simpan jawaban dan konteks hasil retrieval
            pred_answers = []
            retrieved_contexts = []
            latency = []
            real_context=[]

            for question in questions:
                start_time = time.time()
                response = qa.invoke({"query": question})
                end_time = time.time()
                latency.append(end_time - start_time)
                pred_answers.append(response["result"])
                context_docs = response.get("source_documents", [])
                retrieved_contexts.append([doc.page_content for doc in context_docs])
                real_context.append([doc.page_content for doc in qa.retriever.get_real_context()])
                
            # Siapkan dataset untuk evaluasi RAGAS
            ds = Dataset.from_dict({
                "question": questions,
                "answer": pred_answers,
                "ground_truths": ground_truths,
                "contexts": real_context
            })
            run_config = RunConfig(
                timeout=1800,

            )

            # Jalankan evaluasi RAGAS
            evaluator_llm = LangchainLLMWrapper(llm)
            metrics = evaluate(
                ds,
                metrics=[faithfulness_indonesia,answer_relevancy_indonesia,context_relevance_indonesia],
                llm=evaluator_llm,
                embeddings=embedder,
                run_config=run_config
            )
            
            # Format hasil evaluasi
            for i in range(len(questions)):
                results.append({
                    "question": questions[i],
                    "ground_truth": ground_truths[i],
                    "answer": pred_answers[i],
                    "context_with_expansion": "\n---\n".join(retrieved_contexts[i]),
                    "context": "\n---\n".join(real_context[i]),
                    
                    "faithfulness": metrics["faithfulness_indonesia"][i],
                    "answer_relevancy": metrics["answer_relevancy_indonesia"][i],
                    "context_relevance": metrics["context_relevance_indonesia"][i],
                    "latency": latency[i],
                })
                print(len(retrieved_contexts[i]))
            # Simpan ke file CSV
            output_filename = f"evaluation_llm_{llm_type['name']}_embedding_{embedding_type['name']}_k5_nw_3_5.csv"
            df = pd.DataFrame(results)
            df.to_csv(output_filename, index=False)
            print(f"Evaluation complete. Results saved to {output_filename}")


In [22]:
llm_types = [
             {
    "name": "3b",
    "type":"llama3.2:3b",
},
            {
    "name": "1b",
    "type":"llama3.2:1b",
}
    ]
embedding_types = [{
    "name":"indosentencebert",
    "type":"firqaaa/indo-sentence-bert-large",
},
 {
    "name": "3b",
    "type":"llama3.2:3b",
},
             {
    "name": "1b",
    "type":"llama3.2:1b",
}                  
    ]
questions = [
  "Berapa lama pelaksanaan kegiatan magang?",
  "Apa yang dimaksud dengan mitra magang?",
  "Apa yang dimaksud dosen pembimbing magang?",
  "Apa saja kriteria pembimbing lapangan?",
  "Apa saja ketentuan yang harus diperhatikan oleh mahasiswa yang akan melaksanakan magang?",
  "Apa yang dimaksud dengan plagiarisme?",
  "Apa yang dimaksud dengan lampiran?",
  "Apa yang dimaksud dengan Tugas Akhir?",
  "Bagaimana struktur proposal tugas akhir",
  "Apa yang dimaksud dengan daftar pustaka?",
  "Berapa jumlah SKS maksimal yang dapat diambil saat MBKM?",
  "Apa saja yang termasuk kegiatan MBKM?",
  "Apa yang dimaksud dengan MBKM?",
  "Apa tujuan dari kegiatan studi independen?",
  "Apa tujuan pertukaran mahasiswa dalam negeri?",
  "Berapa waktu pelaksanaan KP?",
  "Apa dimaksud dengan kerja praktik?",
  "Apa saja syarat pembimbing laporan KP?",
  "Apa saja berkas yang diperlukan untuk melakukan seminar hasil KP di ITK?",
  "Apa saja kriteria penilaian KP?",
]
ground_truths = [
    "Kegiatan magang dilaksanakan diberbagai mitra selama minimal 6 (enam) bulan dan maksimal 12 (dua belas) bulan",
    "Mitra magang adalah industri/instansi pemerintahan atau swasta/lembaga berbadan hukum yang menerima mahasiswa melaksanakan magang",
    "Dosen pembimbing adalah dosen tetap ITK yang bertugas membimbing peserta magang  secara menyeluruh",
    """
    a. Pembimbing dengan syarat sekurang-kurangnya bergelar Sarjana Strata 1
      dan/atau pengalaman kerja minimal 5 (lima) tahun.
    b. Pernah melakukan supervisi.
    c. Memiliki kemampuan untuk memberikan bimbingan teknis kepada peserta Magang sesuai dengan kompetensinya.
    """,
    """
1. Minimal lulus semester 5 dengan jumlah sks yang telah lulus minimal 100 sks.
2. Pelaksanaan Magang diakui dalam satuan kredit semester (sks).
3. Jumlah sks Magang dapat disetarakan dengan sks mata Kuliah Wajib, Kerja Praktik,
   Tugas Akhir dan/atau mata kuliah pilihan.
4. Pelaksanaan Magang secara teknis akan diatur dalam Kesepakatan Magang antara ITK
yang diwakili program studi Peserta Magang dan Mitra Magang.
    """,
    "Plagiarisme adalah perbuatan sengaja atau tidak sengaja dalam memperoleh atau mencoba memperoleh kredit atau nilai untuk suatu karya ilmiah, dengan mengutip sebagian atau seluruh karya dan atau karya ilmiah pihak lain yang diakui sebagai karya ilmiahnya, tanpa menyatakan sumber secara tepat dan memadai.",
    """
    Lampiran merupakan data atau pelengkap atau hasil olahan yang menunjang penulisan tugas akhir, tetapi tidak dicantumkan di dalam isi tugas
akhir, karena akan mengganggu kesinambungan pembacaan.
    """,
    """
    Tugas Akhir didefinisikan sebagai penulisan karya ilmiah berisi hasil
penelitian menyeluruh yang disusun secara sistematis berdasarkan ketentuan
metode penelitian ilmiah. Penulisan Tugas Akhir ini dimaksudkan sebagai
pelatihan bagi mahasiswa untuk menuangkan gagasannya dalam bentuk sebuah
karya ilmiah""",
    "Proposal TA terdiri dari bagian awal, bagian isi dan bagian akhir.",
    """
    Daftar Pustaka merupakan daftar bacaan yang menjadi sumber, atau
referensi atau acuan dan dasar penulisan tugas akhir. Daftar Pustaka ini dapat
berisi buku, artikel jurnal, majalah, atau surat kabar, wawancara, dan sebagainya.
    """,
    """Jumlah sks maksimal yang dapat diambil dalam rangka pelaksanaan kegiatan
MBKM di program studi yang berbeda di ITK adalah 20 (dua puluh) sks atau
setara 1 semester.""",
    """Ruang lingkup kegiatan MBKM adalah 8 (delapan) kegiatan pembelajaran yang
 meliputi:
 a. Magang/Praktek Kerja;
 b. Membangun Desa/Kuliah Kerja Nyata Tematik (KKNT);
 c. Pertukaran Mahasiswa; d. Proyek Kemanusiaan;
 e. Penelitian/Riset;
 f. Kegiatan Wirausaha;
 g. Studi/Proyek Independen; dan
 h. Asistensi Mengajar di Satuan Pendidikan.""",
  """MBKM adalah
kebijakan Menteri Pendidikan dan Kebudayaan, yang bertujuan mendorong
mahasiswa untuk menguasai berbagai keilmuan yang berguna untuk memasuki
dunia kerja dengan memberikan kesempatan bagi mahasiswa untuk memilih
mata kuliah yang akan mereka ambil.""",
  """Kegiatan studi/proyek independen ditujukan untuk mewujudkan gagasan
 mahasiswa dalam mengembangkan produk inovatif, menyelenggarakan
 pendidikan berbasis riset dan pengembangan serta meningkatkan prestasi
 mahasiswa dalam ajang nasional dan internasional.""",
    """
    Kegiatan pertukaran mahasiswa dalam negeri ditujukan untuk memperluas wawasan mahasiswa, penyetaraan pendidikan, pengkayaan sains dan teknologi (saintek) serta terjadinya kolaborasi invensi dan inovasi multidisiplin lingkup
dalam negeri.
    """,
    """Waktu pelaksanaan KP minimal satu bulan dan maksimal dua bulan, tergantung
pada jenis aktivitas yang ditawarkan oleh mitra KP""",
    """Kerja Praktik (KP) merupakan salah satu bentuk mata kuliah yang wajib ditempuh
oleh mahasiswa Institut Teknologi Kalimantan (ITK) dalam rangka menyelesaikan studi
sesuai Program Studi yang ditempuhnya.""",
    """
    a. Pembimbing KP untuk setiap peserta KP terdiri dari satu orang Dosen
       Pembimbing.
    b. Pembimbing dengan syarat sekurang-kurangnya bergelar Magister (Jabatan
       Akademik Asisten Ahli)
    c. Pembimbing diangkat dan diberhentikan dengan surat keputusan Rektor atas
       usulan Koordinator Program Studi.
    d. Jumlah mahasiswa bimbingan KP mengikuti kebijakan masing-masing prodi.""",
    """
1. Formulir Pendaftaran Seminar Hasil Kerja Praktik (Form. KP-003).
2. Log sheet Mingguan (Form. KP-004).
3. Lembar Konsultasi Bimbingan (Form.KP-005)
4. Lembar Absensi Kehadiran (Form. KP-006)
5. Laporan KP yang sudah disetujui oleh Dosen Pembimbing
6. Surat Keterangan Selesai KP dari Mitra KP (jika ada)
7. Lembar Penilaian dari Pembimbing Lapangan (Form. KP-007)
8. Form Pernyataan Persetujuan Publikasi Laporan Kerja Praktik Mitra KP
(Lampiran 1) (Bagi Mitra KP yang diluar ITK)
    """,
    """
- Performance/Kinerja :
Proses dan hasil kerja dilihat dari hard skill dan soft skill yang dicapai oleh peserta
KP dalam melaksanakan tugasnya sesuai dengan tanggung jawab yang diberikan.
- Presentasi :
Tingkat kemampuan presentasi atau memaparkan hasil penelitian baik kepada
audience/pembimbing lapangan/Dosen Pembimbing saat presentasi hasil KP.
- Poster:
Kemampuan Peserta KP dalam membuat desain, konten, dan informasi tugas
khusus.
- Laporan KP :
Tingkat kualitas laporan KP dari segi tata penulisan dan kelengkapan informasi
substansi tugas khusus
    """,
]
run_ragas_full_eval(file_paths,questions,ground_truths,llm_types, embedding_types)

INFO:root:Initializing embeddings for 'firqaaa/indo-sentence-bert-large' on device: cuda
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: firqaaa/indo-sentence-bert-large


Load LLM llama3.2:3b
Load Embedding firqaaa/indo-sentence-bert-large


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

24
23
29
29
29
31
35
28
28
29
29
18
25
26
25
35
35
32
31
27
Evaluation complete. Results saved to evaluation_llm_3b_embedding_indosentencebert_k5_nw_3_5.csv


INFO:root:Initializing embeddings for 'llama3.2:3b' on device: cuda


...

Load LLM llama3.2:1b
Load Embedding llama3.2:1b


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POS

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POS

35
35
35
35
28
35
35
35
24
35
28
35
35
35
32
35
35
28
35
35
Evaluation complete. Results saved to evaluation_llm_1b_embedding_1b_k5_nw_3_5.csv


#### from pathlib import Path

folder_path = Path(".")

for file in folder_path.glob("evaluation*.csv"):
    


In [ ]:
def calculate_mrr(name, ranks):
    reciprocal_ranks = [(1/r) if r > 0 else 0.0 for r in ranks]
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    return mrr
    
    
    